# Modeling

## 1. Setup

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

### 1.1. Drive mount, paths and extraction

In [4]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/NoteVision')
RAW_DIR = PROJECT_DIR / 'data_raw'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
LOCAL_DIR = Path('/content/primus')
ARCHIVE = RAW_DIR / 'primus.tgz'
EXPECTED_SAMPLES = 87_678
SEED = 42

def count_samples(root):
    n = 0
    for pkg in ('package_aa', 'package_ab'):
        p = root / pkg
        if p.exists():
            n += sum(1 for d in p.iterdir() if d.is_dir() and not d.name.startswith('._'))
    return n

if count_samples(LOCAL_DIR) != EXPECTED_SAMPLES:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $LOCAL_DIR

n = count_samples(LOCAL_DIR)
print('Samples on disk:', n)
assert n == EXPECTED_SAMPLES, 'Incomplete extraction'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Samples on disk: 87678


### 1.2. Preprocessing functions and metrics

In [5]:
IMG_HEIGHT = 128

def load_vocab(vocab_csv):
    """Return (tokens, token_to_id, n_tokens)"""
    df = pd.read_csv(vocab_csv, dtype=str, keep_default_na=False)
    tokens = df['token'].tolist()
    return tokens, {t: i for i, t in enumerate(tokens)}, len(tokens)

def encode_labels(df, token_to_id, max_len):
    """Dense int32 array [N, max_len] (zero padded) and true lengths [N]."""
    labels = np.zeros((len(df), max_len), dtype=np.int32)
    lengths = np.zeros(len(df), dtype=np.int32)
    for i, seq in enumerate(df['label']):
        ids = [token_to_id[t] for t in seq.split()]
        labels[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return labels, lengths

def decode_ids(ids, tokens):
    return ' '.join(tokens[i] for i in ids)

def image_path(root, package, sid):
    return Path(root) / package / sid / f'{sid}.png'

# ... Deleted functions (notebook previo)

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_png(raw, channels=3) # palette (mode P) -> RGB
    gray = tf.image.rgb_to_grayscale(img) # uint8 [H, W, 1]
    h = tf.cast(tf.shape(gray)[0], tf.float32)
    w = tf.cast(tf.shape(gray)[1], tf.float32)
    new_w = tf.maximum(1, tf.cast(tf.round(w * IMG_HEIGHT / h), tf.int32))
    gray = tf.image.resize(tf.cast(gray, tf.float32), (IMG_HEIGHT, new_w),
                           method='bilinear', antialias=True)
    gray = tf.clip_by_value(gray, 0.0, 255.0)
    return 1.0 - gray / 255.0, new_w # ink = 1, background/padding = 0

def build_dataset(df, root, token_to_id, max_label_len, batch_size=32,
                  training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width,
                'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(
        batch_size,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)})
    return ds.prefetch(tf.data.AUTOTUNE)

from collections import Counter, defaultdict

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (ref[i-1] != hyp[j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + (ref[i-1] != hyp[j-1]):
            ops.append(('hit' if ref[i-1] == hyp[j-1] else 'sub', ref[i-1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(('del', ref[i-1]))
            i -= 1
        else:
            ops.append(('ins', None))
            j -= 1
    return d[n][m], ops

def evaluate_sequences(refs, hyps, families):
    tot_ref = tot_err = tot_err_shape = exact = 0
    per_fam = defaultdict(Counter)
    for r, h in zip(refs, hyps):
        dist, ops = align(r, h)
        tot_err += dist
        tot_ref += len(r)
        exact += int(r == h)
        tot_err_shape += align([families[i] for i in r], [families[i] for i in h])[0]
        for op, tok in ops:
            if tok is not None:
                per_fam[families[tok]][op] += 1
    metrics = {'cer': tot_err / tot_ref, 'seq_acc': exact / len(refs),
               'cer_shape': tot_err_shape / tot_ref,
               'n_sequences': len(refs), 'n_tokens': tot_ref}
    rows = []
    for fam, c in per_fam.items():
        support = c['hit'] + c['sub'] + c['del']
        rows.append({'family': fam, 'support': support, 'recall': c['hit'] / support})
    fam_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)
    return metrics, fam_df

# Sanity checks
_fam = ['a', 'a', 'b']
assert align([0, 1, 2], [0, 1, 2])[0] == 0
assert align([0, 1, 2], [0, 2])[0] == 1
assert align([0, 1, 2], [0, 1, 1])[0] == 1
_m, _ = evaluate_sequences([[0, 1, 2]], [[0, 0, 2]], _fam)
assert abs(_m['cer'] - 1/3) < 1e-9 and _m['cer_shape'] == 0.0 and _m['seq_acc'] == 0.0
print('Metric sanity checks passed.')

Metric sanity checks passed.


### 1.3. Bootstrap

In [6]:
parts = {s: pd.read_csv(ARTIFACTS_DIR / f'split_{s}.csv', dtype=str)
         for s in ('train', 'val', 'test')}
for d in parts.values():
    d['seq_len'] = d['seq_len'].astype(int)

tokens, token_to_id, _ = load_vocab(ARTIFACTS_DIR / 'vocab_agnostic.csv')
cfg = json.loads((ARTIFACTS_DIR / 'preprocessing_config.json').read_text())
summ = json.loads((ARTIFACTS_DIR / 'split_summary.json').read_text())

# Guard against drift between notebooks
assert IMG_HEIGHT == cfg['img_height'] and len(tokens) == cfg['vocab_size']
assert {s: len(d) for s, d in parts.items()} == {s: summ[f'n_{s}'] for s in parts}
MAX_LABEL_LEN = cfg['max_label_len']
print('Bootstrap OK:', {s: len(d) for s, d in parts.items()})

Bootstrap OK: {'train': 70138, 'val': 8772, 'test': 8768}


## 2. GPU and Seed

In [7]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(SEED)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. CRNN Model

In [8]:
REDUCTION = 8 # horizontal downsampling
CTC_BLANK = 0 # class 0 = blank; token id i -> class i + 1
NUM_CLASSES = len(tokens) + 1 # 758 tokens + CTC blank
families = [t.split('-')[0] for t in tokens]

class CRNN(keras.Model):
    def __init__(self, num_classes, lstm_units=128, dropout=0.25, **kw):
        super().__init__(**kw)
        filters = (32, 64, 128, 256)
        pools = ((2, 2), (2, 2), (2, 2), (2, 1)) # width /2 three times -> x8; last pool only in height
        self.convs = [layers.Conv2D(f, 3, padding='same', use_bias=False) for f in filters]
        self.bns = [layers.BatchNormalization() for _ in filters]
        self.pools = [layers.MaxPool2D(p) for p in pools]
        self.proj = layers.Dense(256, activation='relu')
        self.drop = layers.Dropout(dropout)
        self.rnn1 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.rnn2 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x, widths = inputs['image'], inputs['width']
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = pool(tf.nn.relu(bn(conv(x), training=training)))
        x = tf.transpose(x, [0, 2, 1, 3])
        s = tf.shape(x)
        x = tf.reshape(x, [s[0], s[1], x.shape[2] * x.shape[3]])
        x = self.drop(self.proj(x), training=training)
        logit_len = widths // REDUCTION
        mask = tf.sequence_mask(logit_len, maxlen=s[1]) # ignore padding in the BiLSTM
        x = self.rnn1(x, mask=mask, training=training)
        x = self.rnn2(self.drop(x, training=training), mask=mask, training=training)
        return self.classifier(self.drop(x, training=training)), logit_len


model = CRNN(NUM_CLASSES)

# Shape smoke test on one real batch
train_ds = build_dataset(parts['train'], LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                         batch_size=16, training=True, seed=SEED)

b = next(iter(train_ds))
logits, logit_len = model({'image': b['image'], 'width': b['width']}, training=False)
print('image:', tuple(b['image'].shape), '| logits:', tuple(logits.shape))
print('logit_len:', logit_len.numpy())
assert logits.shape[1] == b['image'].shape[2] // REDUCTION
assert logits.shape[-1] == NUM_CLASSES
assert bool(tf.reduce_all(logit_len <= logits.shape[1]))
assert bool(tf.reduce_all(logit_len >= b['label_length']))
print('\nTrainable params:', sum(int(np.prod(v.shape)) for v in model.trainable_variables))

image: (16, 128, 1567, 1) | logits: (16, 195, 759)
logit_len: [ 92 110 137 192 129 134 139 165 132 134 113  77 179 195  92 157]

Trainable params: 1896407


## 4. CTC Loss, Training Step and Greedy Decoding

In [9]:
def ctc_loss_fn(labels, label_len, logits, logit_len):
    loss = tf.nn.ctc_loss(labels=labels + 1, # token id i -> class i + 1
                          logits=tf.cast(logits, tf.float32),
                          label_length=label_len, logit_length=logit_len,
                          logits_time_major=False, blank_index=CTC_BLANK)
    return tf.reduce_mean(loss)

def make_steps(model, optimizer):
    """Build tf.functions bound to THIS model and optimizer (call again for each new model)."""
    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32),
        tf.TensorSpec([None, MAX_LABEL_LEN], tf.int32),
        tf.TensorSpec([None], tf.int32)])
    def train_step(image, width, label, label_len):
        with tf.GradientTape() as tape:
            logits, logit_len = model({'image': image, 'width': width}, training=True)
            loss = ctc_loss_fn(label, label_len, logits, logit_len)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32)])
    def infer(image, width):
        return model({'image': image, 'width': width}, training=False)

    return train_step, infer

def greedy_decode(logits, lengths, blank=CTC_BLANK):
    """argmax per step -> collapse repeats -> drop blanks -> back to token ids 0..757."""
    ids = np.argmax(logits, axis=-1)
    out = []
    for seq, n in zip(ids, lengths):
        seq = seq[:n]
        changed = np.concatenate([[True], seq[1:] != seq[:-1]])
        seq = seq[changed]
        out.append((seq[seq != blank] - 1).tolist())
    return out

def predict(batches):
    refs, hyps = [], []
    for b in batches:
        logits, logit_len = infer(b['image'], b['width'])
        hyps += greedy_decode(logits.numpy(), logit_len.numpy())
        lab, ln = b['label'].numpy(), b['label_length'].numpy()
        refs += [lab[i, :ln[i]].tolist() for i in range(len(ln))]
    return refs, hyps

# Decode sanity check (no model): blank=0, token id i = class i + 1
def _fake(classes):
    z = np.zeros((1, len(classes), NUM_CLASSES), dtype=np.float32)
    for t, c in enumerate(classes):
        z[0, t, c] = 1.0
    return z

assert greedy_decode(_fake([0, 3, 3, 0, 5, 5, 0]), [7]) == [[2, 4]]
assert greedy_decode(_fake([3, 0, 3]), [3]) == [[2, 2]] # repeat separated by blank survives
print('Decode sanity checks passed.')

Decode sanity checks passed.


## 5. Overfit Test (64 samples)

In [ ]:
# NOTE: output below is from the previous convention (blank = last class); not re-run.

tf.keras.utils.set_random_seed(SEED)
optimizer = keras.optimizers.Adam(1e-3, clipnorm=5.0)
optimizer.build(model.trainable_variables)
train_step, infer = make_steps(model, optimizer)

small_df = parts['train'].sample(64, random_state=SEED)
small_batches = list(build_dataset(small_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                   batch_size=16, training=False))
EPOCHS = 400

for epoch in range(1, EPOCHS + 1):
    losses = [float(train_step(b['image'], b['width'], b['label'], b['label_length']))
              for b in small_batches]
    if epoch == 1 or epoch % 20 == 0:
        refs, hyps = predict(small_batches)
        m, _ = evaluate_sequences(refs, hyps, families)
        print(f"epoch {epoch:>3} | loss {np.mean(losses):9.3f} | "
              f"CER {m['cer']:.4f} | seq_acc {m['seq_acc']:.3f}")
        if m['seq_acc'] == 1.0:
            break

print('REF:', decode_ids(refs[0], tokens)[:160])
print('HYP:', decode_ids(hyps[0], tokens)[:160])

epoch   1 | loss   665.844 | CER 1.0000 | seq_acc 0.000
epoch  20 | loss   127.497 | CER 1.0000 | seq_acc 0.000
epoch  40 | loss   114.167 | CER 0.9686 | seq_acc 0.000
epoch  60 | loss   105.043 | CER 0.9442 | seq_acc 0.000
epoch  80 | loss    92.656 | CER 0.9587 | seq_acc 0.000
epoch 100 | loss    78.642 | CER 0.9343 | seq_acc 0.000
epoch 120 | loss    62.288 | CER 0.8784 | seq_acc 0.000
epoch 140 | loss    48.297 | CER 0.7912 | seq_acc 0.000
epoch 160 | loss    34.872 | CER 0.7469 | seq_acc 0.000
epoch 180 | loss    23.885 | CER 0.4212 | seq_acc 0.000
epoch 200 | loss    16.597 | CER 0.2693 | seq_acc 0.016
epoch 220 | loss    11.414 | CER 0.1059 | seq_acc 0.188
epoch 240 | loss     7.396 | CER 0.0797 | seq_acc 0.297
epoch 260 | loss     4.860 | CER 0.0326 | seq_acc 0.625
epoch 280 | loss     3.642 | CER 0.0372 | seq_acc 0.578
epoch 300 | loss     2.536 | CER 0.0134 | seq_acc 0.906
epoch 320 | loss     1.945 | CER 0.0145 | seq_acc 0.859
epoch 340 | loss     1.463 | CER 0.0128 | seq_ac

## 6. Short Real-Data Run

### 6.1. Width-Bucketed Pipeline

In [10]:
# Boundaries every 100 px up to 2200, then coarser. Each batch is padded to (boundary - 1),
# so the whole training run sees only ~25 distinct input shapes.
BOUNDS = list(range(600, 2201, 100)) + [2600, 3000, 3600]
MAX_PIXELS = 48000 # batch_size * padded_width budget (~36 images at 1266 px, ~5 GB peak)
MAX_BATCH = 64
BATCH_SIZES = [max(4, min(MAX_BATCH, MAX_PIXELS // b)) for b in BOUNDS] + [4] # last bucket (>= 3600) stays empty
print('Buckets:', len(BOUNDS) + 1, '| batch size per bucket:', BATCH_SIZES)

# Safety: the widest resized image must fit under the top boundary
sz = pd.read_csv(ARTIFACTS_DIR / 'image_sizes.csv')
max_w = int(np.round(sz.width * IMG_HEIGHT / sz.height).max())
assert max_w < BOUNDS[-1], f'widest image ({max_w}) does not fit in the buckets'


def build_dataset_bucketed(df, root, token_to_id, max_label_len, training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width, 'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.bucket_by_sequence_length(
        element_length_func=lambda e: e['width'],
        bucket_boundaries=BOUNDS, bucket_batch_sizes=BATCH_SIZES,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)},
        pad_to_bucket_boundary=True,
        drop_remainder=training) # fixed batch shape in training; keep every sample in val/test
    return ds.prefetch(tf.data.AUTOTUNE)

Buckets: 21 | batch size per bucket: [64, 64, 60, 53, 48, 43, 40, 36, 34, 32, 30, 28, 26, 25, 24, 22, 21, 18, 16, 13, 4]


### 6.2. Input Pipeline Speed

In [11]:
import time
from itertools import islice

short_train = parts['train'].sample(6000, random_state=SEED)
short_val = parts['val'].sample(1000, random_state=SEED)

ds_test = build_dataset_bucketed(short_train, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                 training=True, seed=SEED)
t0, n_img, n_b = time.time(), 0, 0
for b in islice(ds_test, 60):
    n_img += int(b['image'].shape[0]); n_b += 1
dt = time.time() - t0
print(f'{n_b} batches, {n_img} images in {dt:.1f}s -> {n_img/dt:.0f} img/s from the pipeline alone')

60 batches, 2324 images in 20.5s -> 113 img/s from the pipeline alone


### 6.3. Fresh Model and Short Training Run

In [15]:
tf.keras.utils.set_random_seed(SEED)
model = CRNN(NUM_CLASSES)
model({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})  # build weights
optimizer = keras.optimizers.Adam(1e-3, clipnorm=5.0)
optimizer.build(model.trainable_variables)
train_step, infer = make_steps(model, optimizer)

EPOCHS_SHORT = 6
CKPT_DIR = ARTIFACTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
ckpt_path = CKPT_DIR / 'exp00_dryrun_blank0.weights.h5'

train_ds = build_dataset_bucketed(short_train, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                  training=True, seed=SEED)
val_ds = build_dataset_bucketed(short_val, LOCAL_DIR, token_to_id, MAX_LABEL_LEN)

history, best_cer = [], float('inf')
for epoch in range(1, EPOCHS_SHORT + 1):
    t0, losses, n_img = time.time(), [], 0
    for b in train_ds:
        losses.append(float(train_step(b['image'], b['width'], b['label'], b['label_length'])))
        n_img += int(b['image'].shape[0])
    train_time = time.time() - t0

    refs, hyps = predict(val_ds)
    m, _ = evaluate_sequences(refs, hyps, families)

    sec_step = train_time / len(losses)
    est_full_min = sec_step * (len(parts['train']) / (n_img / len(losses))) / 60
    history.append({'epoch': epoch, 'train_loss': np.mean(losses), 'val_cer': m['cer'],
                    'val_seq_acc': m['seq_acc'], 'val_cer_shape': m['cer_shape'],
                    'steps': len(losses), 'sec_per_step': sec_step, 'epoch_sec': train_time})
    print(f"epoch {epoch} | loss {np.mean(losses):8.2f} | val CER {m['cer']:.4f} | "
          f"seq_acc {m['seq_acc']:.3f} | cer_shape {m['cer_shape']:.4f} | "
          f"{len(losses)} steps, {sec_step:.2f} s/step | est. full epoch: {est_full_min:.0f} min")

    if m['cer'] < best_cer:
        best_cer = m['cer']
        model.save_weights(ckpt_path)

pd.DataFrame(history).to_csv(ARTIFACTS_DIR / 'exp00_dryrun_history.csv', index=False)

epoch 1 | loss   155.69 | val CER 0.9809 | seq_acc 0.000 | cer_shape 0.9808 | 146 steps, 1.65 s/step | est. full epoch: 51 min
epoch 2 | loss   127.40 | val CER 0.9362 | seq_acc 0.000 | cer_shape 0.9279 | 146 steps, 1.06 s/step | est. full epoch: 33 min
epoch 3 | loss   120.25 | val CER 0.9394 | seq_acc 0.000 | cer_shape 0.9364 | 146 steps, 1.05 s/step | est. full epoch: 32 min
epoch 4 | loss   104.06 | val CER 0.9786 | seq_acc 0.000 | cer_shape 0.9774 | 146 steps, 1.38 s/step | est. full epoch: 42 min
epoch 5 | loss    64.92 | val CER 0.7072 | seq_acc 0.000 | cer_shape 0.6730 | 146 steps, 0.98 s/step | est. full epoch: 30 min
epoch 6 | loss    33.01 | val CER 0.2957 | seq_acc 0.013 | cer_shape 0.2267 | 146 steps, 0.98 s/step | est. full epoch: 30 min


## 7. EXP-00

### 7.1. Config, schedule and steps per epoch

In [16]:
import math, os, shutil, time
from itertools import islice

CKPT_DIR = ARTIFACTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
RUN_DIR = CKPT_DIR / 'exp00'

class WarmupCosine(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup, then cosine decay to lr_final; a pure function of the step."""
    def __init__(self, peak, final, warmup, total):
        super().__init__()
        self.peak, self.final = float(peak), float(final)
        self.warmup, self.total = int(warmup), int(total)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak * (step + 1.0) / self.warmup
        prog = tf.clip_by_value((step - self.warmup) / float(max(1, self.total - self.warmup)), 0.0, 1.0)
        cos = self.final + 0.5 * (self.peak - self.final) * (1.0 + tf.cos(math.pi * prog))
        return tf.where(step < self.warmup, warm, cos)

    def get_config(self):
        return {'peak': self.peak, 'final': self.final, 'warmup': self.warmup, 'total': self.total}

def compute_steps_per_epoch(df):
    """Exact steps per epoch of the bucketed train pipeline (drop_remainder), from image headers only."""
    cache = ARTIFACTS_DIR / 'train_steps_per_epoch.json'
    sig = {'bounds': BOUNDS, 'batch_sizes': BATCH_SIZES, 'n_train': len(df)}
    if cache.exists():
        c = json.loads(cache.read_text())
        if c['sig'] == sig:
            return c['steps_per_epoch']
    widths = []
    for p, i in zip(df['package'], df['id']):
        with Image.open(image_path(LOCAL_DIR, p, i)) as im:
            w, h = im.size
        widths.append(max(1, int(np.round(np.float32(w) * IMG_HEIGHT / np.float32(h)))))
    bucket = np.searchsorted(BOUNDS, np.array(widths), side='right')
    counts = np.bincount(bucket, minlength=len(BATCH_SIZES))
    steps = int(sum(int(n) // bs for n, bs in zip(counts, BATCH_SIZES)))
    cache.write_text(json.dumps({'sig': sig, 'steps_per_epoch': steps, 'bucket_counts': counts.tolist()}))
    return steps

STEPS_PER_EPOCH = compute_steps_per_epoch(parts['train'])

CFG = {
    'exp_id': 'EXP-00', 'seed': SEED, 'epochs': 8, 'steps_per_epoch': STEPS_PER_EPOCH,
    'lr_peak': 1e-3, 'lr_final': 1e-5, 'warmup_steps': 200, 'clipnorm': 5.0,
    'lstm_units': 128, 'dropout': 0.25, 'max_pixels': MAX_PIXELS, 'max_batch': MAX_BATCH,
    'val_subset_n': 2000, 'early_stop_from_epoch': 4, 'patience': 3,
}

# Fixed validation subset, shared by all experiments
val_sub = parts['val'].sample(CFG['val_subset_n'], random_state=SEED)
ids_path = ARTIFACTS_DIR / 'val_subset_ids.csv'
if ids_path.exists():
    saved = pd.read_csv(ids_path, dtype=str)
    assert saved['id'].tolist() == val_sub['id'].tolist(), 'val subset changed'
else:
    val_sub[['package', 'id']].to_csv(ids_path, index=False)

total = CFG['epochs'] * STEPS_PER_EPOCH
print(f'steps/epoch: {STEPS_PER_EPOCH} | total steps: {total} | '
      f'est. {total * 1.0 / 3600:.1f}-{total * 1.4 / 3600:.1f} h at 1.0-1.4 s/step')

steps/epoch: 1830 | total steps: 14640 | est. 4.1-5.7 h at 1.0-1.4 s/step


### 7.2. Checkpoint helpers

In [17]:
def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1))
    os.replace(tmp, path)

def save_ckpt(run_dir, model, optimizer, state):
    """Write to the alternate slot; the pointer (latest.json) is updated last."""
    run_dir.mkdir(parents=True, exist_ok=True)
    state['n_ckpts'] += 1
    slot = run_dir / f"slot{state['n_ckpts'] % 2}"
    slot.mkdir(exist_ok=True)
    tmp_w = slot / 'tmp.weights.h5'
    model.save_weights(tmp_w)
    os.replace(tmp_w, slot / 'model.weights.h5')
    tmp_o = slot / 'tmp_opt.npz'
    with open(tmp_o, 'wb') as f:
        np.savez(f, **{f'v{i}': v.numpy() for i, v in enumerate(optimizer.variables)})
    os.replace(tmp_o, slot / 'optimizer.npz')
    write_json_atomic(slot / 'state.json', state)
    if hasattr(os, 'sync'):
        os.sync()
    write_json_atomic(run_dir / 'latest.json',
                      {'slot': slot.name, 'global_step': state['global_step'], 'epoch': state['epoch']})

def load_ckpt(run_dir, model, optimizer):
    ptr = run_dir / 'latest.json'
    if not ptr.exists():
        return None
    slot = run_dir / json.loads(ptr.read_text())['slot']
    state = json.loads((slot / 'state.json').read_text())
    model.load_weights(slot / 'model.weights.h5')
    data = np.load(slot / 'optimizer.npz')
    opt_vars = optimizer.variables
    assert len(data.files) == len(opt_vars), 'optimizer variable count mismatch'
    for i, v in enumerate(opt_vars):
        arr = data[f'v{i}']
        assert tuple(v.shape) == arr.shape, f'optimizer variable {i}: shape mismatch'
        v.assign(arr)
    assert int(optimizer.iterations.numpy()) == state['global_step'], 'optimizer.iterations != global_step'
    return state

def save_best(run_dir, model, entry):
    tmp = run_dir / 'tmp_best.weights.h5'
    model.save_weights(tmp)
    os.replace(tmp, run_dir / 'best.weights.h5')
    write_json_atomic(run_dir / 'best.json', entry)

### 7.3. Training loop with resume

In [18]:
def new_state(cfg):
    return {'config': dict(cfg), 'global_step': 0, 'epoch': 1, 'step_in_epoch': 0,
            'epoch_loss_sum': 0.0, 'epoch_loss_n': 0, 'epoch_train_sec': 0.0, 'train_sec_total': 0.0,
            'resume_count': 0, 'n_ckpts': 0, 'best_cer': float('inf'), 'best_epoch': 0,
            'bad_epochs': 0, 'history': [], 'finished': False, 'stop_reason': None}

def run_training(cfg, run_dir, train_df, val_df, ckpt_every=400, max_steps_this_call=None,
                 session_limit_min=None, log_every=200):
    """Fresh start or resume from run_dir. max_steps_this_call simulates a crash (no save)."""
    global infer  # predict() uses the global `infer`; rebind it to this model
    tf.keras.utils.set_random_seed(cfg['seed'])
    model = CRNN(NUM_CLASSES, lstm_units=cfg['lstm_units'], dropout=cfg['dropout'])
    model({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})  # build weights
    total = cfg['epochs'] * cfg['steps_per_epoch']
    sched = WarmupCosine(cfg['lr_peak'], cfg['lr_final'], cfg['warmup_steps'], total)
    opt = keras.optimizers.Adam(sched, clipnorm=cfg['clipnorm'])
    opt.build(model.trainable_variables)

    state = load_ckpt(run_dir, model, opt)
    if state is None:
        state = new_state(cfg)
        print('Starting fresh.')
    else:
        assert state['config'] == dict(cfg), f"config differs from checkpoint: {state['config']}"
        state['resume_count'] += 1
        print(f"Resumed: epoch {state['epoch']}, step_in_epoch {state['step_in_epoch']}, "
              f"global_step {state['global_step']}, lr {float(sched(state['global_step'])):.2e}")
    if state['finished']:
        print('Already finished:', state['stop_reason'])
        return model, opt, state

    train_step, infer = make_steps(model, opt)
    val_ds = build_dataset_bucketed(val_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN)
    t_session, steps_this_call = time.time(), 0

    while state['epoch'] <= cfg['epochs'] and not state['finished']:
        ep = state['epoch']
        ds = build_dataset_bucketed(train_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN, training=True,
                                    seed=cfg['seed'] + 1000 * ep + state['resume_count'])
        remaining = cfg['steps_per_epoch'] - state['step_in_epoch']
        t_last = time.time()
        for b in islice(ds, remaining):
            loss = float(train_step(b['image'], b['width'], b['label'], b['label_length']))
            if not math.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at step {state['global_step']}; last checkpoint kept")
            now = time.time()
            state['epoch_train_sec'] += now - t_last
            state['train_sec_total'] += now - t_last
            state['global_step'] += 1
            state['step_in_epoch'] += 1
            state['epoch_loss_sum'] += loss
            state['epoch_loss_n'] += 1
            steps_this_call += 1
            if state['global_step'] % log_every == 0:
                print(f"  ep {ep} step {state['step_in_epoch']}/{cfg['steps_per_epoch']} "
                      f"(global {state['global_step']}) | loss {loss:8.2f} | "
                      f"lr {float(sched(state['global_step'])):.2e}")
            if state['global_step'] % ckpt_every == 0:
                save_ckpt(run_dir, model, opt, state)
            if max_steps_this_call and steps_this_call >= max_steps_this_call:
                print(f"SIMULATED INTERRUPT at global step {state['global_step']} (nothing saved now)")
                return model, opt, state
            if session_limit_min and (time.time() - t_session) / 60 >= session_limit_min:
                save_ckpt(run_dir, model, opt, state)
                print('Session time limit reached: checkpoint saved, stopping cleanly.')
                return model, opt, state
            t_last = time.time()  # excludes checkpoint time from the training clock

        # ---- end of epoch: validate on the fixed subset ----
        refs, hyps = predict(val_ds)
        m, _ = evaluate_sequences(refs, hyps, families)
        entry = {'epoch': ep, 'global_step': state['global_step'],
                 'train_loss': state['epoch_loss_sum'] / max(1, state['epoch_loss_n']),
                 'val_cer': m['cer'], 'val_seq_acc': m['seq_acc'], 'val_cer_shape': m['cer_shape'],
                 'lr_end': float(sched(state['global_step'])),
                 'epoch_train_min': state['epoch_train_sec'] / 60}
        entry = {k: float(v) for k, v in entry.items()}
        state['history'].append(entry)
        if m['cer'] < state['best_cer']:
            state['best_cer'], state['best_epoch'], state['bad_epochs'] = float(m['cer']), ep, 0
            save_best(run_dir, model, entry)
            tag = ' *best*'
        else:
            state['bad_epochs'] += 1
            tag = f" (no improvement x{state['bad_epochs']})"
        print(f"EPOCH {ep} | loss {entry['train_loss']:.2f} | val CER {m['cer']:.4f} | "
              f"seq_acc {m['seq_acc']:.3f} | cer_shape {m['cer_shape']:.4f} | "
              f"{entry['epoch_train_min']:.1f} min{tag}")
        write_json_atomic(run_dir / 'history.json', state['history'])

        state['epoch'] += 1
        state['step_in_epoch'] = 0
        state['epoch_loss_sum'], state['epoch_loss_n'], state['epoch_train_sec'] = 0.0, 0, 0.0
        if ep >= cfg['early_stop_from_epoch'] and state['bad_epochs'] >= cfg['patience']:
            state['finished'], state['stop_reason'] = True, 'early_stop'
        elif state['epoch'] > cfg['epochs']:
            state['finished'], state['stop_reason'] = True, 'epochs_done'
        save_ckpt(run_dir, model, opt, state)

    print('Finished:', state['stop_reason'])
    return model, opt, state

### 7.4. Resume test

In [19]:
TEST_DIR = CKPT_DIR / 'exp00_resume_test'
if TEST_DIR.exists():
    shutil.rmtree(TEST_DIR)
test_cfg = {**CFG, 'epochs': 3, 'steps_per_epoch': 20, 'val_subset_n': 200}
test_val = val_sub.head(200)

# Call 1: 24 steps = epoch 1 (20) + 4 steps of epoch 2; "crash" right after the step-24 checkpoint
m1, o1, s1 = run_training(test_cfg, TEST_DIR, parts['train'], test_val,
                          ckpt_every=8, max_steps_this_call=24, log_every=8)
ptr = json.loads((TEST_DIR / 'latest.json').read_text())
print('pointer:', ptr)
assert ptr['global_step'] == 24 and ptr['epoch'] == 2

# Restore into a brand-new model/optimizer and compare with the in-memory state at step 24
m2 = CRNN(NUM_CLASSES, lstm_units=CFG['lstm_units'], dropout=CFG['dropout'])
m2({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})
sch2 = WarmupCosine(test_cfg['lr_peak'], test_cfg['lr_final'], test_cfg['warmup_steps'], 3 * 20)
o2 = keras.optimizers.Adam(sch2, clipnorm=test_cfg['clipnorm'])
o2.build(m2.trainable_variables)
st = load_ckpt(TEST_DIR, m2, o2)
assert max(np.abs(a - b).max() for a, b in zip(m1.get_weights(), m2.get_weights())) == 0.0
assert max(np.abs(a.numpy() - b.numpy()).max() for a, b in zip(o1.variables, o2.variables)) == 0.0
assert int(o2.iterations.numpy()) == 24 and st['step_in_epoch'] == 4
print(f"restore OK | iterations {int(o2.iterations.numpy())} | lr {float(sch2(24)):.2e}")

# Call 2: resume mid-epoch and run to the end (epochs 2 and 3, validation, best, history)
m3, o3, s3 = run_training(test_cfg, TEST_DIR, parts['train'], test_val, ckpt_every=8, log_every=10)
assert s3['finished'] and s3['global_step'] == 60 and len(s3['history']) == 3
assert (TEST_DIR / 'best.weights.h5').exists() and (TEST_DIR / 'history.json').exists()
print('Resume test passed.')
shutil.rmtree(TEST_DIR)

Starting fresh.
  ep 1 step 8/20 (global 8) | loss  1163.02 | lr 4.50e-05
  ep 1 step 16/20 (global 16) | loss  1031.19 | lr 8.50e-05
EPOCH 1 | loss 824.65 | val CER 1.0000 | seq_acc 0.000 | cer_shape 1.0000 | 0.4 min *best*
  ep 2 step 4/20 (global 24) | loss   631.41 | lr 1.25e-04
SIMULATED INTERRUPT at global step 24 (nothing saved now)
pointer: {'slot': 'slot0', 'global_step': 24, 'epoch': 2}
restore OK | iterations 24 | lr 1.25e-04
Resumed: epoch 2, step_in_epoch 4, global_step 24, lr 1.25e-04
  ep 2 step 10/20 (global 30) | loss   437.02 | lr 1.55e-04
  ep 2 step 20/20 (global 40) | loss   263.15 | lr 2.05e-04
EPOCH 2 | loss 419.18 | val CER 1.0000 | seq_acc 0.000 | cer_shape 1.0000 | 0.4 min (no improvement x1)
  ep 3 step 10/20 (global 50) | loss   177.53 | lr 2.55e-04
  ep 3 step 20/20 (global 60) | loss   154.88 | lr 3.05e-04
EPOCH 3 | loss 170.88 | val CER 1.0000 | seq_acc 0.000 | cer_shape 1.0000 | 0.4 min (no improvement x2)
Finished: epochs_done
Resume test passed.


### 7.5. Launch EXP-00

In [20]:
model, opt, state = run_training(CFG, RUN_DIR, parts['train'], val_sub,
                                 ckpt_every=400, session_limit_min=None)

Starting fresh.
  ep 1 step 200/1830 (global 200) | loss   124.62 | lr 1.00e-03
  ep 1 step 400/1830 (global 400) | loss   104.99 | lr 1.00e-03
  ep 1 step 600/1830 (global 600) | loss    53.65 | lr 9.98e-04
  ep 1 step 800/1830 (global 800) | loss    20.57 | lr 9.96e-04
  ep 1 step 1000/1830 (global 1000) | loss    11.87 | lr 9.93e-04
  ep 1 step 1200/1830 (global 1200) | loss     6.69 | lr 9.88e-04
  ep 1 step 1400/1830 (global 1400) | loss     4.86 | lr 9.83e-04
  ep 1 step 1600/1830 (global 1600) | loss     5.65 | lr 9.77e-04
  ep 1 step 1800/1830 (global 1800) | loss     6.00 | lr 9.70e-04
EPOCH 1 | loss 58.64 | val CER 0.0317 | seq_acc 0.592 | cer_shape 0.0257 | 29.5 min *best*
  ep 2 step 170/1830 (global 2000) | loss     4.87 | lr 9.63e-04
  ep 2 step 370/1830 (global 2200) | loss     4.40 | lr 9.54e-04
  ep 2 step 570/1830 (global 2400) | loss     2.42 | lr 9.44e-04
  ep 2 step 770/1830 (global 2600) | loss     4.00 | lr 9.34e-04
  ep 2 step 970/1830 (global 2800) | loss     1

### 7.6. Export Results from EXP-00

In [21]:
h = pd.DataFrame(state['history'])
b = h.loc[h['val_cer'].idxmin()]
row = {'exp_id': 'EXP-00', 'date': time.strftime('%Y-%m-%d'), 'epochs_run': len(h),
       'best_epoch': int(b['epoch']), 'lr_schedule': 'warmup 200 + cosine 1e-3 to 1e-5',
       'batch_policy': 'bucketed; 48000 px; max 64', 'val_subset_cer': round(b['val_cer'], 4),
       'seq_acc': round(b['val_seq_acc'], 4), 'cer_shape': round(b['val_cer_shape'], 4),
       'train_time_h': round(state['train_sec_total'] / 3600, 2),
       'notes': 'Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4'}
print(pd.DataFrame([row]).to_csv(index=False))

exp_id,date,epochs_run,best_epoch,lr_schedule,batch_policy,val_subset_cer,seq_acc,cer_shape,train_time_h,notes
EXP-00,2026-11-01,8,8,warmup 200 + cosine 1e-3 to 1e-5,bucketed; 48000 px; max 64,0.0073,0.8695,0.006,3.94,Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4

